# RAGTIME2 Mini Data Viewer

A compact notebook for inspecting the public RAGTIME2 dataset without downloading the full collection. It uses the Hugging Face Dataset Viewer API to fetch small bounded samples and display them as tables.

The optional final cell shows how to call the private RAGTIME retrieval API when `RAGTIME_BASE_URL` and `RAGTIME_BEARER_TOKEN` are available. Configured private responses use a private Hugging Face Bucket as durable L2 and an owner-only disposable `simplecache` directory as L1; cache operational failures warn once and degrade to a cache miss or unsuccessful write, while typed payload validation remains strict.

## Setup

Run with the repo dependencies using `uv run jupyter lab notebooks/ragtime2_mini_viewer.ipynb`, or execute headlessly with `uv run jupyter nbconvert --execute --to notebook --inplace notebooks/ragtime2_mini_viewer.ipynb`.

Adjust the parameters below to inspect another config, split, offset, or number of rows. Keep `ROWS_TO_LOAD` small; the full collection is large.

In [1]:
from textwrap import shorten

import pandas as pd
import requests
from IPython.display import Markdown, display
from trec_ragtime import RagtimeClient, RagtimeConfigurationError

DATASET = "trec-ragtime/ragtime2"
HF_API = "https://datasets-server.huggingface.co"
ROWS_TO_LOAD = 5
REQUEST_TIMEOUT_SECONDS = 20

ORIGINAL_CONFIG = "default"
ORIGINAL_SPLIT = "eng"

MT_CONFIG = "mt_docs"
MT_SPLIT = "spa.mt.eng"

## Helpers

In [2]:
def fetch_dataset_viewer(endpoint, **params):
    """Fetch a bounded payload from the Hugging Face Dataset Viewer API."""
    try:
        response = requests.get(
            f"{HF_API}/{endpoint}",
            params=params,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        response.raise_for_status()
    except requests.RequestException as exc:
        return None, f"unavailable: {exc.__class__.__name__}: {exc}"

    return response.json(), "live"


def rows_to_frame(payload):
    rows = [entry.get("row", {}) for entry in payload.get("rows", [])]
    return pd.DataFrame(rows)


def preview_text(value, width=280):
    if not isinstance(value, str):
        return value
    return shorten(" ".join(value.split()), width=width, placeholder="...")


def display_source_status(label, status):
    display(Markdown(f"**{label} source:** `{status}`"))

## Dataset Metadata

This cell reads split and size metadata. It is lightweight compared with downloading the dataset itself.

In [3]:
splits_payload, splits_status = fetch_dataset_viewer("splits", dataset=DATASET)
size_payload, size_status = fetch_dataset_viewer("size", dataset=DATASET)

display_source_status("Splits", splits_status)
if splits_payload:
    splits_df = pd.DataFrame(splits_payload.get("splits", []))
    display(splits_df)
else:
    display(Markdown("No split metadata loaded. Run this notebook with outbound network access to query Hugging Face."))

display_source_status("Size", size_status)
if size_payload:
    split_sizes = pd.DataFrame(size_payload.get("size", {}).get("splits", []))
    display(split_sizes[["config", "split", "num_rows", "num_bytes_parquet_files"]])
else:
    display(Markdown("No size metadata loaded. Run this notebook with outbound network access to query Hugging Face."))

**Splits source:** `live`

,dataset,config,split
0,trec-ragtime/ragtime2,default,spa
1,trec-ragtime/ragtime2,default,rus
2,trec-ragtime/ragtime2,default,zho
3,trec-ragtime/ragtime2,default,eng
4,trec-ragtime/ragtime2,mt_docs,spa.mt.eng
5,trec-ragtime/ragtime2,mt_docs,rus.mt.eng
6,trec-ragtime/ragtime2,mt_docs,zho.mt.eng


**Size source:** `live`

,config,split,num_rows,num_bytes_parquet_files
0,default,spa,1000095,1805855890
1,default,rus,1000095,1526278793
2,default,zho,1000095,1608686319
3,default,eng,1000095,1988053237
4,mt_docs,spa.mt.eng,1000095,1790956973
5,mt_docs,rus.mt.eng,1000095,1168839556
6,mt_docs,zho.mt.eng,1000095,1726848832


## Original-Language Document Sample

The default sample uses the English split from the original-language config.

In [4]:
original_payload, original_status = fetch_dataset_viewer(
    "rows",
    dataset=DATASET,
    config=ORIGINAL_CONFIG,
    split=ORIGINAL_SPLIT,
    offset=0,
    length=ROWS_TO_LOAD,
)

display_source_status(f"{ORIGINAL_CONFIG}/{ORIGINAL_SPLIT}", original_status)
if original_payload:
    original_df = rows_to_frame(original_payload)
    original_preview = original_df.assign(text_preview=original_df["text"].map(preview_text)).drop(columns=["text"])
    display(original_preview)
else:
    original_df = pd.DataFrame()
    display(Markdown("No original-language rows loaded."))

**default/eng source:** `live`

,id,url,date,text_preview
0,0e4ae416-c558-4741-9e4a-4988869bcd74_2154817,https://www.timesunion.com/sports/article/Mean...,2021-08-01T02:08:10.000Z,"Means, Franco lead Orioles to 5-2 win over Tig..."
1,0e4ae416-c558-4741-9e4a-4988869bcd74_56357083,https://nanoreview.net/en/phone-compare/samsun...,2021-08-01T02:08:10.000Z,Home > Smartphone comparison > Galaxy M30s vs ...
2,0e4ae416-c558-4741-9e4a-4988869bcd74_74872923,https://lionsroarnews.com/26185/aame-summit-br...,2021-08-01T02:08:10.000Z,AAME Summit brings people together at Southeas...
3,0e4ae416-c558-4741-9e4a-4988869bcd74_90201977,https://www.eastbaytimes.com/2021/07/31/233-st...,2021-08-01T02:08:10.000Z,Breaking News PUBLISHED: | UPDATED: At two maj...
4,0e4ae416-c558-4741-9e4a-4988869bcd74_106796673,https://nanoreview.net/en/phone-compare/samsun...,2021-08-01T02:08:10.000Z,Home > Smartphone comparison > Galaxy S9 vs Ga...


## Machine-Translated Document Sample

The translated config exposes English machine translations for non-English source documents. The default sample uses Spanish-to-English rows.

In [5]:
mt_payload, mt_status = fetch_dataset_viewer(
    "rows",
    dataset=DATASET,
    config=MT_CONFIG,
    split=MT_SPLIT,
    offset=0,
    length=ROWS_TO_LOAD,
)

display_source_status(f"{MT_CONFIG}/{MT_SPLIT}", mt_status)
if mt_payload:
    mt_df = rows_to_frame(mt_payload)
    mt_preview = mt_df.assign(text_preview=mt_df["text"].map(preview_text))
    visible_columns = [column for column in ["id", "lang", "source", "created", "text_preview"] if column in mt_preview.columns]
    display(mt_preview[visible_columns])
else:
    mt_df = pd.DataFrame()
    display(Markdown("No machine-translated rows loaded."))

**mt_docs/spa.mt.eng source:** `live`

,id,lang,source,created,text_preview
0,0e4ae416-c558-4741-9e4a-4988869bcd74_1793347,eng,cc-news_2021-08,2021-08-01T02:08:19.000Z,Border agent and civilian killed in a crash in...
1,0e4ae416-c558-4741-9e4a-4988869bcd74_3828405,eng,cc-news_2021-08,2021-08-01T02:08:30.000Z,State Transit Removes Minatitlán Outposts · Gl...
2,0e4ae416-c558-4741-9e4a-4988869bcd74_4048167,eng,cc-news_2021-08,2021-08-01T02:08:32.000Z,Former Haitian Official Found Killed in RD · G...
3,0e4ae416-c558-4741-9e4a-4988869bcd74_7328224,eng,cc-news_2021-08,2021-08-01T02:08:55.000Z,Advertising Advertising Advertising # Está InT...
4,0e4ae416-c558-4741-9e4a-4988869bcd74_11588636,eng,cc-news_2021-08,2021-08-01T02:09:42.000Z,They recover stolen items in the south of the ...


## Optional Private Retrieval API

Registered RAGTIME participants can set `RAGTIME_BASE_URL`, `RAGTIME_BEARER_TOKEN`, and the private artifact settings (`RAGTIME_HF_BUCKET_ID`, optional `RAGTIME_ARTIFACT_PREFIX`, and optional `RAGTIME_ARTIFACT_CACHE_DIR`) in the environment before running this cell. The cell is skipped when credentials are missing; cache or transport failures warn once and degrade to a cache miss or unsuccessful write, while typed payload validation remains strict.

In [ ]:
try:
    with RagtimeClient.from_env() as ragtime_client:
        api_result = ragtime_client.search("Avengers Endgame Avatar highest grossing")
        artifact_configuration = ragtime_client.cache_configuration
except RagtimeConfigurationError:
    display(Markdown("Private retrieval API credentials are not set; skipping API call."))
else:
    display(pd.DataFrame(api_result.as_records()).head(ROWS_TO_LOAD))
    cache_status = "hit" if api_result.local_cache_hit else "miss"
    display(Markdown(f"**Private artifact cache:** `{cache_status}` — `{artifact_configuration}`"))